# 3. Seed variance and ablations

**Read this before believing any improvement in notebook 2.** The same
configuration trained at three seeds gives a spread; a difference between two
single runs is inside noise unless it exceeds `sqrt(2) x sd`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
import matplotlib.pyplot as plt
pd.set_option("display.width", 130)


In [ ]:
T = '../results/tables/'
noise = pd.read_csv(T+'seed_variance.csv')
noise[noise.split=='test_id'].round(5).to_string(index=False)

The `diff_noise_scale` column is the bar every claimed gain has to clear.

In [ ]:
runs = pd.read_csv(T+'seed_runs.csv')
piv = runs[runs.split=='test_id'].pivot(index='metric', columns='seed', values='value')
piv.round(5)

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists('../results/figures/seed_noise.png'):
    display(Image(filename='../results/figures/seed_noise.png'))

## Ablations: one mechanism removed at a time

`no_message_passing` is the one that isolates the graph. Its parameter count is
matched to the full model on purpose, so the comparison is about message
passing rather than about capacity.

In [ ]:
abl = pd.read_csv(T+'ablations.csv')
p = abl[abl.split=='test_id'].set_index('variant')[['params','mae','spearman_within_scenario','train_seconds']]
p.round(4)

In [ ]:
st = pd.read_csv(T+'ablation_statistics.csv')
st[st.split=='test_id'][['name_a','mean_a','mean_b','difference','p_adjusted','significant']].round(5).to_string(index=False)

### Placing each ablation against the noise scale

A significant paired p-value is not enough: the delta also has to be large
relative to seed-to-seed variation of the identical configuration.

In [ ]:
from sndsur.metrics.stats import noise_scale, noise_verdict
ns = noise_scale('mae', runs[(runs.split=='test_id')&(runs.metric=='mae')].value.values)
full = float(abl[(abl.split=='test_id')&(abl.variant=='full')].mae.iloc[0])
rows = []
for _, r in abl[abl.split=='test_id'].iterrows():
    if r.variant == 'full': continue
    gain = float(r.mae) - full  # positive = removing it hurt
    ratio, verdict = noise_verdict(gain, ns)
    rows.append({'variant': r.variant, 'mae': round(float(r.mae),5),
                 'delta_vs_full': round(gain,5), 'ratio_to_noise': round(ratio,2),
                 'verdict': verdict})
pd.DataFrame(rows)